# 01j - Detector CLIP (features congeladas) — generalização cross-generator

Abordagem de **Ojha et al., CVPR 2023** (*Towards Universal Fake Image Detectors that Generalize Across Generative Models*). Em vez de fine-tunar uma CNN — que vicia na digital do gerador de treino —, classifica num espaço de features **não treinado para real-vs-fake**: o **CLIP-ViT congelado** + um probe linear.

Treina o probe em **140k** (real = CelebA, fake = StyleGAN) e avalia **por gerador** no ArtiFact (geradores não vistos, metade `test`), comparando com o teto ~0.62 do CNN fine-tunado (01i).

> Requer `open_clip_torch` (`pip install open_clip_torch`). O ViT-L/14 baixa ~1.7 GB na 1ª vez. Embeddings são cacheados em disco.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import open_clip

sys.path.insert(0, str(Path.cwd().parent / "notebooks_140k"))
from aug_utils import artifact_split

PROJECT_ROOT = Path.cwd().resolve().parent
_envf = PROJECT_ROOT / "data_root.env"
DATA_ROOT = Path(_envf.read_text().strip()) if _envf.exists() else PROJECT_ROOT / "data"
RAW_140K     = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
ARTIFACT_DIR = DATA_ROOT / "raw" / "artifact_faces"
EMB_DIR  = PROJECT_ROOT / "artifacts" / "clip_detector"
FIGS_DIR = PROJECT_ROOT / "reports" / "figures"
EMB_DIR.mkdir(parents=True, exist_ok=True); FIGS_DIR.mkdir(parents=True, exist_ok=True)

CLIP_MODEL, PRETRAINED = "ViT-L-14", "openai"
N_TRAIN_PER_CLASS = 2000
N_REAL_TEST, N_PER_GEN = 800, 200
BATCH, SEED = 64, 42
rng = np.random.default_rng(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()
print("Device:", DEVICE, "| CLIP:", CLIP_MODEL, PRETRAINED)

## 1. Dados — treino (140k) e teste por gerador (ArtiFact, metade test)

In [ ]:
def imgs_in(folder):
    fs = []
    for e in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        fs += list(Path(folder).glob(e))
    return sorted(fs)

def sample(files, n):
    files = list(files)
    return files if n >= len(files) else [files[i] for i in rng.choice(len(files), n, replace=False)]

def by_source(folder):
    g = {}
    for p in imgs_in(folder):
        s = p.name.split("__")[0] if "__" in p.name else "?"
        g.setdefault(s, []).append(p)
    return {k: sorted(v) for k, v in g.items()}

train_real = sample(imgs_in(RAW_140K / "train" / "real"), N_TRAIN_PER_CLASS)
train_fake = sample(imgs_in(RAW_140K / "train" / "fake"), N_TRAIN_PER_CLASS)

fake_groups, real_groups = by_source(ARTIFACT_DIR / "fake"), by_source(ARTIFACT_DIR / "real")
GENERATORS = sorted(fake_groups)

art_real_pool = []
for s in sorted(real_groups):
    art_real_pool += artifact_split(real_groups[s], which="test")
art_real_test = sample(art_real_pool, N_REAL_TEST)
art_fake_test = {g: sample(artifact_split(fake_groups[g], which="test"), N_PER_GEN) for g in GENERATORS}

print(f"treino 140k: {len(train_real)} real + {len(train_fake)} fake (StyleGAN)")
print(f"teste ArtiFact: {len(art_real_test)} real + {len(GENERATORS)} geradores x {N_PER_GEN}")

## 2. Embeddings CLIP (congelado, com cache)

`encode_image` do CLIP, normalizado. Salva em `.npy` para re-rodar sem re-extrair.

In [ ]:
@torch.no_grad()
def embed(paths):
    out = []
    for i in range(0, len(paths), BATCH):
        ims = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths[i:i + BATCH]]).to(DEVICE)
        f = model.encode_image(ims).float()
        f = f / f.norm(dim=-1, keepdim=True)
        out.append(f.cpu().numpy())
    return np.concatenate(out)

def cached(name, paths):
    fp = EMB_DIR / f"{CLIP_MODEL}_{name}_n{len(paths)}.npy"
    if fp.exists():
        return np.load(fp)
    e = embed(paths); np.save(fp, e)
    return e

Xtr_real = cached("train_real", train_real)
Xtr_fake = cached("train_fake", train_fake)
Xte_real = cached("art_real_test", art_real_test)
Xte_fake = {g: cached(f"art_{g}", art_fake_test[g]) for g in GENERATORS}
print("dim embedding:", Xtr_real.shape[1])

## 3. Probe linear real-vs-fake (treinado só no StyleGAN)

In [ ]:
Xtr = np.concatenate([Xtr_real, Xtr_fake])
ytr = np.r_[np.ones(len(Xtr_real)), np.zeros(len(Xtr_fake))]   # real=1, fake=0 (igual ao 01i)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
clf.fit(Xtr, ytr)

def auc_real_vs(Xfake):
    X = np.concatenate([Xte_real, Xfake]); y = np.r_[np.ones(len(Xte_real)), np.zeros(len(Xfake))]
    return roc_auc_score(y, clf.predict_proba(X)[:, 1])

clip_auc = {g: auc_real_vs(Xte_fake[g]) for g in GENERATORS}
clip_mean = float(np.mean(list(clip_auc.values())))
print("AUC CLIP por gerador (real-vs-gerador, geradores nao vistos):\n")
for g in sorted(clip_auc, key=lambda k: -clip_auc[k]):
    print(f"  {g:20s} {clip_auc[g]:.3f}")
print(f"\n  MEAN = {clip_mean:.4f}")

## 4. Comparação — CLIP vs CNN fine-tunado (01i)

In [ ]:
cnn_auc = {}
p = PROJECT_ROOT / "artifacts" / "cross_gen_aug" / "results.json"
if p.exists():
    r = json.loads(p.read_text())
    xs = [x for x in r if x["recipe"] == "jpeg+noise"]
    for g in GENERATORS:
        if xs and g in xs[0]:
            cnn_auc[g] = float(np.mean([x[g] for x in xs]))

rows = [{"gerador": g, "CNN_jpeg+noise": round(cnn_auc.get(g, float("nan")), 3), "CLIP_probe": round(clip_auc[g], 3),
         "delta": round(clip_auc[g] - cnn_auc.get(g, float("nan")), 3)} for g in GENERATORS]
df = pd.DataFrame(rows).sort_values("CLIP_probe", ascending=False)
if cnn_auc:
    df.loc[len(df)] = ["MEAN", round(np.mean(list(cnn_auc.values())), 3), round(clip_mean, 3),
                       round(clip_mean - np.mean(list(cnn_auc.values())), 3)]
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 4.5))
g_order = [r["gerador"] for r in rows]
x = np.arange(len(g_order)); w = 0.38
if cnn_auc:
    ax.bar(x - w/2, [cnn_auc.get(g, 0) for g in g_order], w, label="CNN (jpeg+noise)", color="#a0aec0")
ax.bar(x + w/2, [clip_auc[g] for g in g_order], w, label="CLIP probe", color="#3182ce")
ax.axhline(0.5, color="red", ls=":", lw=1, label="chance")
ax.set_xticks(x); ax.set_xticklabels(g_order, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("AUC real-vs-gerador"); ax.set_ylim(0, 1.0)
ax.set_title("Cross-generator: CNN fine-tunado vs CLIP congelado + probe linear")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS_DIR / "clip_vs_cnn.png", dpi=130, bbox_inches="tight")
plt.show()

(EMB_DIR / "clip_results.json").write_text(json.dumps(
    {"clip_model": CLIP_MODEL, "clip_auc": clip_auc, "clip_mean": clip_mean, "cnn_auc": cnn_auc}, indent=2))

## 5. Leitura e próximos passos

- Se o CLIP **sobe os geradores que o CNN errava** (face_synthetics, stable_diffusion, projected_gan), confirma Ojha: features semânticas genéricas generalizam onde o fine-tuning vicia na digital do StyleGAN.
- Se subir só os GANs, o ganho é parcial e vale combinar (CLIP + sinal de resíduo/cor).

**Próximos:** vizinho-mais-próximo em vez de probe linear (variante principal do paper); modelo CLIP maior; avaliar no conjunto de teste final; e, se promissor, este é o detector que vai pro relatório como abordagem generalizável.